<a href="https://colab.research.google.com/github/6pb4wnww4g-beep/Jason/blob/main/Jason_94.3B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# --- JASON DEVELOPMENT 94.3B ---
# Endringer fra 93.3:
# - Betting_Data ark lagt til
# - TAR_TDR ark lagt til
# - Team_Data utvidet med API Attack / API Defence
# - Ingen endring i PS, xP eller The Observer
# - 94.2B: Fixer manglende xMin L12-hjelpefunksjoner + Pred GI basert på GI-share.
# - 94.2C: Legger til FDR_Overview med motstander, H/A og FPL FDR-farger.
# - 94.3: Min 6 tilbake til ekte L6, GI Share/Pred GI på L19, Pred GI Δ og BPS/90 tilbake i TO.

import requests
import pandas as pd
import numpy as np
from google.colab import auth
import gspread
from google.auth import default
import urllib3
import math
import time

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)
sh = gc.open("Jason Development")


def clean_float(val):
    try:
        return float(val)
    except:
        return 0.0


def get_fdr_multiplier_v87(fdr_score):
    f = round(fdr_score)
    return (
        1.28 if f <= 1 else
        1.20 if f == 2 else
        1.00 if f == 3 else
        0.88 if f == 4 else
        0.82
    )


def calculate_progressive_xp_v87(ps_val, avg_min):
    if avg_min < 30 or ps_val <= 0:
        return 0.0
    return round(max(0.0, (ps_val * 0.10)), 1)


def calculate_xmin_factor_v94(xmin):
    """
    xMin factor for Jason 94.0B2.
    80 = nÃ¸ytral. Under 60 straffes hardt.
    """
    try:
        x = max(0.0, min(90.0, float(xmin)))
    except:
        return 1.0

    if x >= 90:
        return 1.06
    if x >= 80:
        return 1.00
    if x >= 70:
        return 0.90
    if x >= 60:
        return 0.75
    if x >= 45:
        return 0.45
    return 0.25


def apply_xmin_to_ps_v94(ps_val, xmin):
    try:
        return float(ps_val) * calculate_xmin_factor_v94(xmin)
    except:
        return ps_val


def read_xmin_overrides_v94(sh):
    try:
        ws = sh.worksheet("xMin_Override")
        df = pd.DataFrame(ws.get_all_records())
    except:
        return {}

    if df.empty or "Player ID" not in df.columns or "xMin Override" not in df.columns:
        return {}

    out = {}
    for _, r in df.iterrows():
        try:
            pid = str(r.get("Player ID", "")).strip()
            raw = r.get("xMin Override", "")
            if pid and raw != "":
                out[pid] = max(0.0, min(90.0, float(str(raw).replace(",", "."))))
        except:
            pass
    return out


def normalize_team_key_v94(name):
    """
    Robust lagmatching mellom FPL short_name og Market-ark.
    Returnerer helst FPL short_name.
    """
    s = str(name).strip()

    aliases = {
        "ARS": "ARS", "Arsenal": "ARS",
        "AVL": "AVL", "Aston Villa": "AVL",
        "BOU": "BOU", "Bournemouth": "BOU",
        "BRE": "BRE", "Brentford": "BRE",
        "BHA": "BHA", "Brighton": "BHA", "Brighton and Hove Albion": "BHA",
        "CHE": "CHE", "Chelsea": "CHE",
        "CRY": "CRY", "Crystal Palace": "CRY",
        "EVE": "EVE", "Everton": "EVE",
        "FUL": "FUL", "Fulham": "FUL",
        "IPS": "IPS", "Ipswich": "IPS", "Ipswich Town": "IPS",
        "LEE": "LEE", "Leeds": "LEE", "Leeds United": "LEE",
        "LEI": "LEI", "Leicester": "LEI", "Leicester City": "LEI",
        "LIV": "LIV", "Liverpool": "LIV",
        "MCI": "MCI", "Man City": "MCI", "Manchester City": "MCI",
        "MUN": "MUN", "Man Utd": "MUN", "Man United": "MUN", "Manchester United": "MUN",
        "NEW": "NEW", "Newcastle": "NEW", "Newcastle United": "NEW",
        "NFO": "NFO", "Nottingham Forest": "NFO", "Nottm Forest": "NFO", "Nott'm Forest": "NFO",
        "SOU": "SOU", "Southampton": "SOU",
        "SUN": "SUN", "Sunderland": "SUN",
        "TOT": "TOT", "Spurs": "TOT", "Tottenham": "TOT", "Tottenham Hotspur": "TOT",
        "WHU": "WHU", "West Ham": "WHU", "West Ham United": "WHU",
        "WOL": "WOL", "Wolves": "WOL", "Wolverhampton Wanderers": "WOL",
        "COV": "COV", "Coventry": "COV", "Coventry City": "COV",
        "HUL": "HUL", "Hull": "HUL", "Hull City": "HUL",
    }

    return aliases.get(s, s)


def read_market_match_lookup_v94():
    """
    Leser Market_Match_Strength_v2_1 og lager lookup per Team.

    94.0F FIX:
    Vi bruker get_all_values() i stedet for get_all_records().
    Årsak: get_all_records/numericise kan tolke norsk desimalkomma feil:
        2,6345  -> 26345
        51,7709 -> 517709

    Med get_all_values() får vi ren tekst og parser selv.
    """
    sheet_name = "Market_Match_Strength_v2_1"

    try:
        ws = sh.worksheet(sheet_name)
        values = ws.get_all_values()
    except Exception:
        print(f"Fant ikke {sheet_name}. Market-kolonner settes til 0.")
        return {}

    if not values or len(values) < 2:
        print(f"{sheet_name} er tomt. Market-kolonner settes til 0.")
        return {}

    headers = values[0]
    rows = values[1:]
    market_df = pd.DataFrame(rows, columns=headers)

    required = [
        "Team",
        "Team Att",
        "Match Att",
        "Team xG",
        "Team Def",
        "Match Def",
        "CS%"
    ]

    missing = [c for c in required if c not in market_df.columns]
    if missing:
        print(f"{sheet_name} mangler kolonner: {missing}. Market-kolonner settes til 0.")
        return {}

    if "Kickoff" in market_df.columns:
        market_df = market_df.sort_values("Kickoff")

    def sf(v):
        """
        Robust Market-float parser.

        Fikser norsk Google Sheets-format:
        84,3374 -> 84.3374
        2,3866  -> 2.3866
        51,7709 -> 51.7709

        Hvis både komma og punktum finnes, antar vi at komma er tusenskiller.
        """
        if v is None:
            return 0.0

        s = str(v).strip()
        if s == "":
            return 0.0

        s = s.replace(" ", "")

        if "," in s and "." not in s:
            s = s.replace(",", ".")
        elif "," in s and "." in s:
            s = s.replace(",", "")

        try:
            return float(s)
        except:
            return 0.0

    lookup = {}

    for _, r in market_df.iterrows():
        team_key = normalize_team_key_v94(r.get("Team", ""))

        if team_key in lookup:
            continue

        lookup[team_key] = {
            "Team Att": round(sf(r.get("Team Att", 0)), 2),
            "Match Att": round(sf(r.get("Match Att", 0)), 2),
            "xG": round(sf(r.get("Team xG", 0)), 2),
            "Team Def": round(sf(r.get("Team Def", 0)), 2),
            "Match Def": round(sf(r.get("Match Def", 0)), 2),
            "CS%": round(sf(r.get("CS%", 0)), 1),
            "Opp xG": round(sf(r.get("Opp xG", 0)), 2)
        }

    print(f"Market-kobling: {len(lookup)} lag lest fra {sheet_name}.")

    for dbg in ["ARS", "MCI", "LIV"]:
        if dbg in lookup:
            print(f"DEBUG MARKET {dbg}: {lookup[dbg]}")

    return lookup

def calculate_pred_gi_v94(team_xg, player_gi, team_total_gi, xmin_factor):
    """
    Jason 94.2 Pred GI.

    Pred GI =
        Team xG
        * (spiller GI / lagets totale GI)
        * xMin Factor
    """
    try:
        txg = max(0.0, float(team_xg))
        gi = max(0.0, float(player_gi))
        tgi = max(0.0, float(team_total_gi))
        xf = max(0.0, float(xmin_factor))

        if tgi <= 0:
            return 0.0

        return round(txg * (gi / tgi) * xf, 2)
    except:
        return 0.0


def calculate_gi_share_v94(player_gi, team_total_gi):
    try:
        gi = max(0.0, float(player_gi))
        tgi = max(0.0, float(team_total_gi))
        if tgi <= 0:
            return 0.0
        return round(gi / tgi, 4)
    except:
        return 0.0


def read_xmin_history_archive_v94():
    """
    Leser xMin_History_Archive hvis den finnes.
    Nøkkel: (Player ID, Round) -> Minutes.
    """
    sheet_name = "xMin_History_Archive"

    try:
        ws = sh.worksheet(sheet_name)
        df = pd.DataFrame(ws.get_all_records())
    except Exception:
        return {}

    if df.empty or "Player ID" not in df.columns or "Round" not in df.columns or "Minutes" not in df.columns:
        return {}

    archive = {}

    for _, r in df.iterrows():
        try:
            pid = str(r.get("Player ID", "")).strip()
            rnd = int(float(str(r.get("Round", "")).replace(",", ".")))
            mins = float(str(r.get("Minutes", 0)).replace(",", "."))
            if pid != "":
                archive[(pid, rnd)] = mins
        except:
            continue

    return archive


def update_xmin_history_archive_worksheet(rows):
    """
    Skriver xMin_History_Archive når vi har en nesten/full sesong.
    Beskytter arket mot å bli overskrevet tidlig i ny sesong.
    """
    if not rows:
        return

    try:
        max_round = max([
            int(r.get("Round", 0))
            for r in rows
            if str(r.get("Round", "")).strip() != ""
        ])
    except:
        return

    if max_round < 36:
        print("xMin_History_Archive beholdes uendret: nåværende history har ikke GW36 ennå.")
        return

    sheet_name = "xMin_History_Archive"
    df = pd.DataFrame(rows)

    cols = ["Player ID", "Name", "Club", "Pos", "Round", "Minutes"]

    for c in cols:
        if c not in df.columns:
            df[c] = ""

    df = df[cols].sort_values(["Player ID", "Round"]).reset_index(drop=True)

    print(f"Oppdaterer {sheet_name} med {len(df)} history-rader...")

    try:
        ws = sh.worksheet(sheet_name)
        ws.clear()
    except Exception:
        ws = sh.add_worksheet(
            title=sheet_name,
            rows=str(max(1000, len(df) + 20)),
            cols="20"
        )

    ws.resize(
        rows=max(1000, len(df) + 20),
        cols=max(10, len(cols) + 3)
    )

    ws.update([df.columns.tolist()] + df.values.tolist())
    time.sleep(5)


def history_minutes_for_rounds_v94(hist_df, rounds):
    if hist_df is None or hist_df.empty:
        return 0.0

    if "round" not in hist_df.columns or "minutes" not in hist_df.columns:
        return 0.0

    try:
        return float(hist_df[hist_df["round"].isin(rounds)]["minutes"].sum())
    except:
        return 0.0


def archive_minutes_for_rounds_v94(player_id, archive_lookup, rounds):
    pid = str(player_id)
    total = 0.0

    for rnd in rounds:
        try:
            total += float(archive_lookup.get((pid, int(rnd)), 0.0))
        except:
            continue

    return total


def calculate_auto_xmin_l12_v94(player_id, hist_full, next_gw, archive_lookup):
    """
    Jason 94.2B xMin Auto.

    Fast regel:
    - GW37 og GW38 brukes aldri i bootstrap-grunnlaget.
    - Alltid 12 kamper i divisor.
    - GW1:  GW25-36 fra forrige sesong.
    - GW2:  GW26-36 fra forrige sesong + ny GW1.
    - GW3:  GW27-36 fra forrige sesong + ny GW1-2.
    - ...
    - GW13: ny GW1-12.
    - Etter GW13: rullerende siste 12 spilte GW i ny sesong.

    Formel:
    xMin Auto = minutter i vinduet / 12
    """
    try:
        ngw = int(next_gw)
    except:
        ngw = 1

    if ngw <= 12:
        prev_start = 25 + (ngw - 1)
        prev_rounds = list(range(prev_start, 37))
        current_rounds = list(range(1, ngw))
    elif ngw == 13:
        prev_rounds = []
        current_rounds = list(range(1, 13))
    else:
        prev_rounds = []
        current_rounds = list(range(ngw - 12, ngw))

    prev_minutes_archive = archive_minutes_for_rounds_v94(player_id, archive_lookup, prev_rounds)
    prev_minutes_history = history_minutes_for_rounds_v94(hist_full, prev_rounds)

    prev_minutes = prev_minutes_archive if prev_minutes_archive > 0 else prev_minutes_history
    current_minutes = history_minutes_for_rounds_v94(hist_full, current_rounds)

    window_minutes = prev_minutes + current_minutes
    auto_xmin = min(90.0, max(0.0, window_minutes / 12.0))

    if prev_rounds and current_rounds:
        window_label = f"Prev GW{prev_rounds[0]}-{prev_rounds[-1]} + GW{current_rounds[0]}-{current_rounds[-1]}"
    elif prev_rounds:
        window_label = f"Prev GW{prev_rounds[0]}-{prev_rounds[-1]}"
    elif current_rounds:
        window_label = f"GW{current_rounds[0]}-{current_rounds[-1]}"
    else:
        window_label = "No window"

    return round(auto_xmin, 1), round(window_minutes, 1), window_label


def read_gi_history_archive_v94():
    """
    Leser GI_History_Archive hvis den finnes.
    Nøkkel: (Player ID, Round) -> GI.
    """
    sheet_name = "GI_History_Archive"

    try:
        ws = sh.worksheet(sheet_name)
        df = pd.DataFrame(ws.get_all_records())
    except Exception:
        return {}

    if df.empty or "Player ID" not in df.columns or "Round" not in df.columns or "GI" not in df.columns:
        return {}

    archive = {}

    for _, r in df.iterrows():
        try:
            pid = str(r.get("Player ID", "")).strip()
            rnd = int(float(str(r.get("Round", "")).replace(",", ".")))
            gi = float(str(r.get("GI", 0)).replace(",", "."))
            if pid != "":
                archive[(pid, rnd)] = gi
        except:
            continue

    return archive


def update_gi_history_archive_worksheet(rows):
    """
    Skriver GI_History_Archive når vi har en nesten/full sesong.
    Beskytter arket mot å bli overskrevet tidlig i ny sesong.
    """
    if not rows:
        return

    try:
        max_round = max([
            int(r.get("Round", 0))
            for r in rows
            if str(r.get("Round", "")).strip() != ""
        ])
    except:
        return

    if max_round < 36:
        print("GI_History_Archive beholdes uendret: nåværende history har ikke GW36 ennå.")
        return

    sheet_name = "GI_History_Archive"
    df = pd.DataFrame(rows)

    cols = ["Player ID", "Name", "Club", "Pos", "Round", "Goals", "Assists", "GI"]

    for c in cols:
        if c not in df.columns:
            df[c] = ""

    df = df[cols].sort_values(["Player ID", "Round"]).reset_index(drop=True)

    print(f"Oppdaterer {sheet_name} med {len(df)} history-rader...")

    try:
        ws = sh.worksheet(sheet_name)
        ws.clear()
    except Exception:
        ws = sh.add_worksheet(
            title=sheet_name,
            rows=str(max(1000, len(df) + 20)),
            cols="20"
        )

    ws.resize(
        rows=max(1000, len(df) + 20),
        cols=max(10, len(cols) + 3)
    )

    ws.update([df.columns.tolist()] + df.values.tolist())
    time.sleep(5)


def history_gi_for_rounds_v94(hist_df, rounds):
    if hist_df is None or hist_df.empty:
        return 0.0

    if "round" not in hist_df.columns:
        return 0.0

    if "goals_scored" not in hist_df.columns or "assists" not in hist_df.columns:
        return 0.0

    try:
        h = hist_df[hist_df["round"].isin(rounds)]
        return float(
            pd.to_numeric(h["goals_scored"], errors="coerce").fillna(0).sum() +
            pd.to_numeric(h["assists"], errors="coerce").fillna(0).sum()
        )
    except:
        return 0.0


def archive_gi_for_rounds_v94(player_id, archive_lookup, rounds):
    pid = str(player_id)
    total = 0.0

    for rnd in rounds:
        try:
            total += float(archive_lookup.get((pid, int(rnd)), 0.0))
        except:
            continue

    return total


def get_l19_rounds_v94(next_gw):
    """
    L19-vindu for GI Share / Pred GI.

    GW1:  forrige GW20-38.
    GW2:  forrige GW21-38 + ny GW1.
    ...
    GW19: forrige GW38 + ny GW1-18.
    GW20: ny GW1-19.
    Etter GW20: rullerende siste 19 GW i ny sesong.
    """
    try:
        ngw = int(next_gw)
    except:
        ngw = 1

    if ngw <= 19:
        prev_start = 20 + (ngw - 1)
        prev_rounds = list(range(prev_start, 39))
        current_rounds = list(range(1, ngw))
    elif ngw == 20:
        prev_rounds = []
        current_rounds = list(range(1, 20))
    else:
        prev_rounds = []
        current_rounds = list(range(ngw - 19, ngw))

    return prev_rounds, current_rounds


def calculate_player_gi_l19_v94(player_id, hist_full, next_gw, gi_archive_lookup):
    prev_rounds, current_rounds = get_l19_rounds_v94(next_gw)

    prev_gi_archive = archive_gi_for_rounds_v94(player_id, gi_archive_lookup, prev_rounds)
    prev_gi_history = history_gi_for_rounds_v94(hist_full, prev_rounds)

    prev_gi = prev_gi_archive if prev_gi_archive > 0 else prev_gi_history
    current_gi = history_gi_for_rounds_v94(hist_full, current_rounds)

    return round(prev_gi + current_gi, 2)


def calculate_min6_real_v94(player_id, hist_full, next_gw, minutes_archive_lookup, fixtures_r, team_id):
    """
    Min 6 i TO:
    faktiske minutter i siste 6 relevante GW / antall kamper laget spilte.
    I tidlig bootstrap brukes forrige sesong GW31-36, deretter fases ny sesong inn.
    """
    try:
        ngw = int(next_gw)
    except:
        ngw = 1

    if ngw <= 6:
        prev_start = 31 + (ngw - 1)
        prev_rounds = list(range(prev_start, 37))
        current_rounds = list(range(1, ngw))
    else:
        prev_rounds = []
        current_rounds = list(range(ngw - 6, ngw))

    prev_minutes_archive = archive_minutes_for_rounds_v94(player_id, minutes_archive_lookup, prev_rounds)
    prev_minutes_history = history_minutes_for_rounds_v94(hist_full, prev_rounds)

    prev_minutes = prev_minutes_archive if prev_minutes_archive > 0 else prev_minutes_history
    current_minutes = history_minutes_for_rounds_v94(hist_full, current_rounds)

    rounds = prev_rounds + current_rounds
    team_matches = [
        f for f in fixtures_r
        if f.get("event") in rounds and
        (
            f.get("team_h") == team_id or
            f.get("team_a") == team_id
        )
    ]

    if len(team_matches) <= 0:
        return 0.0

    return round((prev_minutes + current_minutes) / len(team_matches), 1)



def get_l6_rounds_v94(next_gw):
    """
    L6-vindu for Form, P6, PPM LAST 6, GI(6), xGI(6), GI/90(6), xGI/90(6) og Heat/deltaer.

    Bootstrap ved sesongstart:
    - GW1:  forrige sesong GW31-36
    - GW2:  forrige sesong GW32-36 + ny GW1
    - GW3:  forrige sesong GW33-36 + ny GW1-2
    - GW4:  forrige sesong GW34-36 + ny GW1-3
    - GW5:  forrige sesong GW35-36 + ny GW1-4
    - GW6:  forrige sesong GW36 + ny GW1-5
    - GW7+: rullerende siste 6 GW i ny sesong

    Dette gjør at L6-kolonnene ikke blir 0 ved next_gw=1.
    """
    try:
        ngw = int(next_gw)
    except:
        ngw = 1

    if ngw <= 6:
        prev_start = 31 + (ngw - 1)
        prev_rounds = list(range(prev_start, 37))
        current_rounds = list(range(1, ngw))
    else:
        prev_rounds = []
        current_rounds = list(range(ngw - 6, ngw))

    return prev_rounds, current_rounds, prev_rounds + current_rounds


def update_xmin_override_worksheet(df_f):
    """
    Bygger xMin_Override som live arbeidsark.

    Denne versjonen bruker semikolon i Google Sheets-formlene:
    =IF(LEN(F2)>0;F2;G2)

    Faktorformelen bruker brÃ¸k som 106/100 i stedet for 1.06.
    Da unngÃ¥r vi problemer med norsk/engelsk desimalskilletegn.
    """
    existing = read_xmin_overrides_v94(sh)
    work = df_f.copy()

    if "Player ID" not in work.columns:
        return

    work["Player ID"] = work["Player ID"].astype(str)
    work["xMin Override"] = work["Player ID"].map(existing).fillna("")

    cols = [
        "Player ID", "Navn", "Lag", "Pos", "Pris",
        "xMin Override", "xMin Auto", "xMin L12 Min", "xMin Window",
        "xMin Final", "xMin Factor",
        "PS1 Before xMin", "PS1 After xMin",
        "PS(5) Before xMin", "PS(5) After xMin",
        "PS Tot Before xMin", "PS Tot After xMin",
        "PPK", "P6", "Min S/6", "Min-guide", "Status", "Merknad"
    ]

    for c in cols:
        if c not in work.columns:
            work[c] = ""

    rows = []
    current_row = 1

    for pos, limit in [("DEF", 50), ("MID", 50), ("FWD", 50), ("GKP", 25)]:
        block = (
            work[work["Pos"] == pos]
            .sort_values("PS Tot After xMin", ascending=False)
            .head(limit)
        )

        rows.append(cols)
        current_row += 1

        for _, r in block.iterrows():
            rn = current_row

            # F = xMin Override
            # G = xMin Auto
            # H = xMin L12 Min
            # I = xMin Window
            # J = xMin Final
            # K = xMin Factor
            # L = PS1 Before
            # N = PS(5) Before
            # P = PS Tot Before

            xfinal_formula = f'=IF(LEN(F{rn})>0;F{rn};G{rn})'
            xfactor_formula = (
                f'=IF(J{rn}="";"";'
                f'IF(J{rn}>=90;106/100;'
                f'IF(J{rn}>=80;1;'
                f'IF(J{rn}>=70;90/100;'
                f'IF(J{rn}>=60;75/100;'
                f'IF(J{rn}>=45;45/100;25/100))))))'
            )

            row = [
                r.get("Player ID", ""),
                r.get("Navn", ""),
                r.get("Lag", ""),
                r.get("Pos", ""),
                r.get("Pris", ""),
                r.get("xMin Override", ""),
                r.get("xMin Auto", ""),
                r.get("xMin L12 Min", ""),
                r.get("xMin Window", ""),
                xfinal_formula,
                xfactor_formula,
                r.get("PS1 Before xMin", ""),
                f'=L{rn}*K{rn}',
                r.get("PS(5) Before xMin", ""),
                f'=N{rn}*K{rn}',
                r.get("PS Tot Before xMin", ""),
                f'=P{rn}*K{rn}',
                r.get("PPK", ""),
                r.get("P6", ""),
                r.get("Min S/6", ""),
                r.get("Min-guide", ""),
                r.get("Status", ""),
                r.get("Merknad", "")
            ]

            rows.append(row)
            current_row += 1

        rows.append([""] * len(cols))
        current_row += 1

    print("Oppdaterer xMin_Override...")
    try:
        ws = sh.worksheet("xMin_Override")
        ws.clear()
    except Exception:
        ws = sh.add_worksheet(title="xMin_Override", rows="1000", cols="40")

    ws.resize(rows=max(300, len(rows) + 20), cols=max(25, len(cols) + 5))
    ws.update(rows, value_input_option="USER_ENTERED")
    time.sleep(5)


def trend_symbol(diff, threshold):
    if diff > threshold:
        return "ð"
    elif diff < -threshold:
        return "ð"
    else:
        return "â"


def level_symbol(val):
    if val >= 0.80:
        return "ð¢"
    elif val >= 0.45:
        return "ð¡"
    else:
        return "ð´"


def combined_trend_symbol(last6, season, threshold):
    level = level_symbol(season)
    diff = last6 - season
    trend = trend_symbol(diff, threshold)
    return f"{level}{trend}"


def reset_conditional_formatting(ws):
    try:
        metadata = sh.fetch_sheet_metadata()
        rule_count = 0

        for sheet in metadata.get("sheets", []):
            if sheet["properties"]["sheetId"] == ws.id:
                rule_count = len(sheet.get("conditionalFormats", []))
                break

        requests_list = []
        for _ in range(rule_count):
            requests_list.append({
                "deleteConditionalFormatRule": {
                    "sheetId": ws.id,
                    "index": 0
                }
            })

        if requests_list:
            sh.batch_update({"requests": requests_list})
    except:
        pass


def update_worksheet(sheet_name, dataframe):
    print(f"Oppdaterer {sheet_name}...")

    df_clean = dataframe.replace(
        [np.inf, -np.inf],
        np.nan
    ).fillna(0)

    try:
        ws = sh.worksheet(sheet_name)
        ws.clear()
    except:
        ws = sh.add_worksheet(
            title=sheet_name,
            rows="2000",
            cols="120"
        )

    ws.update(
        [df_clean.columns.tolist()] +
        df_clean.values.tolist()
    )

    time.sleep(5)


def get_or_create_worksheet(sheet_name, rows="2000", cols="120"):
    try:
        ws = sh.worksheet(sheet_name)
    except:
        ws = sh.add_worksheet(
            title=sheet_name,
            rows=rows,
            cols=cols
        )
    return ws


def col_to_a1(col_num):
    result = ""
    while col_num:
        col_num, remainder = divmod(col_num - 1, 26)
        result = chr(65 + remainder) + result
    return result


def apply_observer_formatting(ws, df, data_ranges):
    reset_conditional_formatting(ws)

    columns = df.columns.tolist()
    requests_list = []

    column_widths = {
        "Name": 180, "Club": 55, "Pos": 55, "Price": 65, "xMin": 65,
        "xP": 65, "PS(5)": 70,
        "PS1": 60, "PS2": 60, "PS3": 60, "PS4": 60, "PS5": 60,
        "Team Att": 75, "Match Att": 80, "xG": 60, "Pred GI": 75, "Pred GI Δ": 75,
        "GI Share": 70, "Team GI": 70,
        "Team Def": 75, "Match Def": 80, "CS%": 60,
        "Heat": 75, "GIÎ": 60, "xGIÎ": 65,
        "ICT/90": 70, "ICTÎ": 60,
        "Form": 65, "PPM LAST 6": 90, "P6": 55, "PPM": 60,
        "BxGI/90": 75,
        "Mins guide": 95, "Min 6": 65,
        "GI": 50, "xGI": 60,
        "GI (6)": 65, "xGI (6)": 70,
        "Pot": 55,
        "GI/90 (6)": 75, "xGI/90 (6)": 85,
        "GI/90 (S)": 75, "xGI/90 (S)": 85
    }

    for idx, col in enumerate(columns):
        requests_list.append({
            "updateDimensionProperties": {
                "range": {
                    "sheetId": ws.id,
                    "dimension": "COLUMNS",
                    "startIndex": idx,
                    "endIndex": idx + 1
                },
                "properties": {"pixelSize": column_widths.get(col, 70)},
                "fields": "pixelSize"
            }
        })

    for block in data_ranges:
        header_row_index = block["header_row"] - 1

        requests_list.append({
            "repeatCell": {
                "range": {
                    "sheetId": ws.id,
                    "startRowIndex": header_row_index,
                    "endRowIndex": header_row_index + 1,
                    "startColumnIndex": 0,
                    "endColumnIndex": len(columns)
                },
                "cell": {
                    "userEnteredFormat": {
                        "backgroundColor": {
                            "red": 1.0,
                            "green": 1.0,
                            "blue": 1.0
                        },
                        "textFormat": {
                            "bold": True,
                            "foregroundColor": {
                                "red": 0.0,
                                "green": 0.0,
                                "blue": 0.0
                            }
                        },
                        "horizontalAlignment": "CENTER"
                    }
                },
                "fields": "userEnteredFormat(backgroundColor,textFormat,horizontalAlignment)"
            }
        })

    # BlÃ¥: lav = hvit, hÃ¸y = #337fe5
    blue_white_cols = [
        "xP",
        "Form",
        "P6",
        "BxGI/90",
        "GI (6)",
        "GI/90 (6)",
        "GI/90 (S)",
        "Team Att",
        "Match Att",
        "xG",
        "Pred GI",
        "Pred GI Δ",
        "GI Share",
        "Team GI",
        "Team Def",
        "Match Def",
        "CS%"
    ]

    for col_name in blue_white_cols:
        if col_name in columns:
            c_idx = columns.index(col_name)

            for block in data_ranges:
                if block["num_rows"] <= 0:
                    continue

                requests_list.append({
                    "addConditionalFormatRule": {
                        "rule": {
                            "ranges": [{
                                "sheetId": ws.id,
                                "startRowIndex": block["data_start"] - 1,
                                "endRowIndex": block["data_end"],
                                "startColumnIndex": c_idx,
                                "endColumnIndex": c_idx + 1
                            }],
                            "gradientRule": {
                                "minpoint": {
                                    "color": {
                                        "red": 1.0,
                                        "green": 1.0,
                                        "blue": 1.0
                                    },
                                    "type": "MIN"
                                },
                                "maxpoint": {
                                    "color": {
                                        "red": 0.2,
                                        "green": 0.498,
                                        "blue": 0.898
                                    },
                                    "type": "MAX"
                                }
                            }
                        },
                        "index": 0
                    }
                })

    # PS: lav = lys gul 2, hÃ¸y = #57bb8a
    ps_cols = ["PS(5)", "PS1", "PS2", "PS3", "PS4", "PS5"]

    for col_name in ps_cols:
        if col_name in columns:
            c_idx = columns.index(col_name)

            for block in data_ranges:
                if block["num_rows"] <= 0:
                    continue

                requests_list.append({
                    "addConditionalFormatRule": {
                        "rule": {
                            "ranges": [{
                                "sheetId": ws.id,
                                "startRowIndex": block["data_start"] - 1,
                                "endRowIndex": block["data_end"],
                                "startColumnIndex": c_idx,
                                "endColumnIndex": c_idx + 1
                            }],
                            "gradientRule": {
                                "minpoint": {
                                    "color": {
                                        "red": 1.0,
                                        "green": 0.949,
                                        "blue": 0.800
                                    },
                                    "type": "MIN"
                                },
                                "maxpoint": {
                                    "color": {
                                        "red": 0.341,
                                        "green": 0.733,
                                        "blue": 0.541
                                    },
                                    "type": "MAX"
                                }
                            }
                        },
                        "index": 0
                    }
                })

    # ICT/90: lav = lys gul, hÃ¸y = dus lilla
    if "ICT/90" in columns:
        c_idx = columns.index("ICT/90")

        for block in data_ranges:
            if block["num_rows"] <= 0:
                continue

            requests_list.append({
                "addConditionalFormatRule": {
                    "rule": {
                        "ranges": [{
                            "sheetId": ws.id,
                            "startRowIndex": block["data_start"] - 1,
                            "endRowIndex": block["data_end"],
                            "startColumnIndex": c_idx,
                            "endColumnIndex": c_idx + 1
                        }],
                        "gradientRule": {
                            "minpoint": {
                                "color": {
                                    "red": 1.0,
                                    "green": 0.949,
                                    "blue": 0.800
                                },
                                "type": "MIN"
                            },
                            "maxpoint": {
                                "color": {
                                    "red": 0.710,
                                    "green": 0.494,
                                    "blue": 0.863
                                },
                                "type": "MAX"
                            }
                        }
                    },
                    "index": 0
                }
            })

    if requests_list:
        sh.batch_update({"requests": requests_list})


def update_observer_worksheet(dataframe):
    print("Oppdaterer The Observer...")

    start_row = 16

    df_clean = dataframe.replace(
        [np.inf, -np.inf],
        np.nan
    ).fillna(0)

    ws = get_or_create_worksheet(
        "The Observer",
        rows="2000",
        cols="80"
    )

    last_col_letter = col_to_a1(len(df_clean.columns))
    clear_range = f"A{start_row}:{last_col_letter}2000"
    ws.batch_clear([clear_range])

    blocks = [
        ("DEF", 21),
        ("MID", 31),
        ("FWD", 21),
        ("GKP", 11)
    ]

    all_values = []
    data_ranges = []
    current_row = start_row

    for pos, limit in blocks:
        block_df = (
            df_clean[df_clean["Pos"] == pos]
            .sort_values("xP", ascending=False)
            .head(limit)
        )

        all_values.append(df_clean.columns.tolist())

        header_row = current_row
        data_start = current_row + 1

        rows = block_df.values.tolist()

        for row in rows:
            all_values.append(row)

        data_end = data_start + len(rows) - 1

        data_ranges.append({
            "pos": pos,
            "header_row": header_row,
            "data_start": data_start,
            "data_end": data_end,
            "num_rows": len(rows)
        })

        current_row = data_end + 2
        all_values.append([""] * len(df_clean.columns))

    ws.update(
        f"A{start_row}",
        all_values
    )

    apply_observer_formatting(
        ws,
        df_clean,
        data_ranges
    )

    time.sleep(5)


def update_fdr_overview_worksheet(fixtures_r, teams_lookup, next_gw):
    """
    Lager og formaterer FDR_Overview.

    Viser de 5 neste GW-ene Jason bruker i PS:
    Team | GWx | GWx+1 | GWx+2 | GWx+3 | GWx+4

    Celleformat:
    OPP (H/A) FDR

    Farger:
    FDR 1 = mørk grønn
    FDR 2 = lys grønn
    FDR 3 = grå
    FDR 4 = lys rød
    FDR 5 = mørk rød

    Håndterer også blank og double gameweek.
    """
    sheet_name = "FDR_Overview"
    target_gws = list(range(int(next_gw), int(next_gw) + 5))

    rows = [["Team"] + [f"GW{gw}" for gw in target_gws]]
    fdr_matrix = []

    teams_sorted = sorted(
        teams_lookup.items(),
        key=lambda x: str(x[1])
    )

    for team_id, team_short in teams_sorted:
        row = [team_short]
        fdr_row = []

        for gw in target_gws:
            matches = [
                f for f in fixtures_r
                if f.get("event") == gw and
                (
                    f.get("team_h") == team_id or
                    f.get("team_a") == team_id
                )
            ]

            if not matches:
                row.append("-")
                fdr_row.append(None)
                continue

            labels = []
            fdr_values = []

            for f in matches:
                is_home = f.get("team_h") == team_id

                opp_id = f.get("team_a") if is_home else f.get("team_h")
                opp = teams_lookup.get(opp_id, "")

                ha = "H" if is_home else "A"

                fdr = (
                    f.get("team_h_difficulty")
                    if is_home
                    else f.get("team_a_difficulty")
                )

                try:
                    fdr_num = int(round(float(fdr)))
                except:
                    fdr_num = None

                if fdr_num is None:
                    labels.append(f"{opp} ({ha})")
                else:
                    labels.append(f"{opp} ({ha}) {fdr_num}")
                    fdr_values.append(fdr_num)

            row.append(" / ".join(labels))

            if fdr_values:
                fdr_row.append(round(sum(fdr_values) / len(fdr_values), 2))
            else:
                fdr_row.append(None)

        rows.append(row)
        fdr_matrix.append(fdr_row)

    try:
        ws = sh.worksheet(sheet_name)
        ws.clear()
    except Exception:
        ws = sh.add_worksheet(
            title=sheet_name,
            rows=str(max(50, len(rows) + 20)),
            cols="12"
        )

    ws.resize(
        rows=max(50, len(rows) + 20),
        cols=max(8, len(rows[0]) + 2)
    )

    ws.update(rows)
    time.sleep(2)

    # Fjern eksisterende conditional formatting på dette arket.
    reset_conditional_formatting(ws)

    requests_list = []

    # Header-format
    requests_list.append({
        "repeatCell": {
            "range": {
                "sheetId": ws.id,
                "startRowIndex": 0,
                "endRowIndex": 1,
                "startColumnIndex": 0,
                "endColumnIndex": len(rows[0])
            },
            "cell": {
                "userEnteredFormat": {
                    "backgroundColor": {"red": 1.0, "green": 1.0, "blue": 1.0},
                    "textFormat": {
                        "bold": True,
                        "foregroundColor": {"red": 0.0, "green": 0.0, "blue": 0.0}
                    },
                    "horizontalAlignment": "CENTER"
                }
            },
            "fields": "userEnteredFormat(backgroundColor,textFormat,horizontalAlignment)"
        }
    })

    # Team-kolonne
    requests_list.append({
        "updateDimensionProperties": {
            "range": {
                "sheetId": ws.id,
                "dimension": "COLUMNS",
                "startIndex": 0,
                "endIndex": 1
            },
            "properties": {"pixelSize": 70},
            "fields": "pixelSize"
        }
    })

    # GW-kolonner
    for c in range(1, len(rows[0])):
        requests_list.append({
            "updateDimensionProperties": {
                "range": {
                    "sheetId": ws.id,
                    "dimension": "COLUMNS",
                    "startIndex": c,
                    "endIndex": c + 1
                },
                "properties": {"pixelSize": 120},
                "fields": "pixelSize"
            }
        })

    # FPL-lignende FDR-farger
    def fdr_color(fdr):
        if fdr is None:
            return {"red": 1.0, "green": 1.0, "blue": 1.0}

        try:
            f = float(fdr)
        except:
            return {"red": 1.0, "green": 1.0, "blue": 1.0}

        if f <= 1.5:
            return {"red": 0.18, "green": 0.67, "blue": 0.42}  # FDR 1
        if f <= 2.5:
            return {"red": 0.62, "green": 0.86, "blue": 0.65}  # FDR 2
        if f <= 3.5:
            return {"red": 0.86, "green": 0.86, "blue": 0.86}  # FDR 3
        if f <= 4.5:
            return {"red": 0.95, "green": 0.55, "blue": 0.55}  # FDR 4
        return {"red": 0.74, "green": 0.18, "blue": 0.18}      # FDR 5

    for r_idx, fdr_row in enumerate(fdr_matrix, start=1):
        # Team-kolonnen
        requests_list.append({
            "repeatCell": {
                "range": {
                    "sheetId": ws.id,
                    "startRowIndex": r_idx,
                    "endRowIndex": r_idx + 1,
                    "startColumnIndex": 0,
                    "endColumnIndex": 1
                },
                "cell": {
                    "userEnteredFormat": {
                        "textFormat": {"bold": True},
                        "horizontalAlignment": "CENTER"
                    }
                },
                "fields": "userEnteredFormat(textFormat,horizontalAlignment)"
            }
        })

        for c_idx, fdr in enumerate(fdr_row, start=1):
            requests_list.append({
                "repeatCell": {
                    "range": {
                        "sheetId": ws.id,
                        "startRowIndex": r_idx,
                        "endRowIndex": r_idx + 1,
                        "startColumnIndex": c_idx,
                        "endColumnIndex": c_idx + 1
                    },
                    "cell": {
                        "userEnteredFormat": {
                            "backgroundColor": fdr_color(fdr),
                            "horizontalAlignment": "CENTER"
                        }
                    },
                    "fields": "userEnteredFormat(backgroundColor,horizontalAlignment)"
                }
            })

    if requests_list:
        sh.batch_update({"requests": requests_list})

    print(f"Oppdaterte {sheet_name} med GW{target_gws[0]}-{target_gws[-1]}.")


# --- BETTING / TAR-TDR INFRASTRUKTUR 93.4 ---

def safe_round(value, digits=1):
    try:
        if pd.isna(value):
            return 0
        return round(float(value), digits)
    except:
        return 0


def normalize_0_100(value, min_value, max_value):
    try:
        value = float(value)
        min_value = float(min_value)
        max_value = float(max_value)

        if max_value == min_value:
            return 50.0

        normalized = (
            (value - min_value) /
            (max_value - min_value)
        ) * 100

        return round(
            max(
                0.0,
                min(100.0, normalized)
            ),
            1
        )
    except:
        return 50.0


def build_betting_data_from_fixtures(fixtures_r, teams_lookup):
    betting_archive = []

    for f in fixtures_r:

        if not f.get("event"):
            continue

        betting_archive.append({

            "GW":
                f.get("event", ""),

            "Date":
                f.get("kickoff_time", ""),

            "HomeTeam":
                teams_lookup.get(
                    f.get("team_h"),
                    ""
                ),

            "AwayTeam":
                teams_lookup.get(
                    f.get("team_a"),
                    ""
                ),

            "HomeOdds":
                "",

            "DrawOdds":
                "",

            "AwayOdds":
                "",

            "Source":
                "Manual / football-data.co.uk later"
        })

    return betting_archive


def build_tar_tdr_from_api(
    fixtures_r,
    teams_lookup,
    team_strength_lookup
):
    raw_rows = []

    for f in fixtures_r:

        if not f.get("event"):
            continue

        home_id = f.get("team_h")
        away_id = f.get("team_a")

        home = teams_lookup.get(home_id, "")
        away = teams_lookup.get(away_id, "")

        h = team_strength_lookup.get(home_id, {})
        a = team_strength_lookup.get(away_id, {})

        # API-logikk:
        # TAR = eget angrep - motstanders forsvar
        # TDR = eget forsvar - motstanders angrep

        home_api_tar_raw = (
            clean_float(h.get("attack_home")) -
            clean_float(a.get("defence_away"))
        )

        away_api_tar_raw = (
            clean_float(a.get("attack_away")) -
            clean_float(h.get("defence_home"))
        )

        home_api_tdr_raw = (
            clean_float(h.get("defence_home")) -
            clean_float(a.get("attack_away"))
        )

        away_api_tdr_raw = (
            clean_float(a.get("defence_away")) -
            clean_float(h.get("attack_home"))
        )

        raw_rows.append({

            "GW":
                f.get("event", ""),

            "Team":
                home,

            "Opponent":
                away,

            "H/A":
                "H",

            "API_TAR_raw":
                home_api_tar_raw,

            "Market_TAR":
                "",

            "TAR":
                "",

            "API_TDR_raw":
                home_api_tdr_raw,

            "Market_TDR":
                "",

            "TDR":
                ""
        })

        raw_rows.append({

            "GW":
                f.get("event", ""),

            "Team":
                away,

            "Opponent":
                home,

            "H/A":
                "A",

            "API_TAR_raw":
                away_api_tar_raw,

            "Market_TAR":
                "",

            "TAR":
                "",

            "API_TDR_raw":
                away_api_tdr_raw,

            "Market_TDR":
                "",

            "TDR":
                ""
        })

    if not raw_rows:
        return []

    tar_values = [
        r["API_TAR_raw"]
        for r in raw_rows
    ]

    tdr_values = [
        r["API_TDR_raw"]
        for r in raw_rows
    ]

    tar_min = min(tar_values)
    tar_max = max(tar_values)

    tdr_min = min(tdr_values)
    tdr_max = max(tdr_values)

    tar_tdr_archive = []

    for r in raw_rows:

        api_tar = normalize_0_100(
            r["API_TAR_raw"],
            tar_min,
            tar_max
        )

        api_tdr = normalize_0_100(
            r["API_TDR_raw"],
            tdr_min,
            tdr_max
        )

        tar_tdr_archive.append({

            "GW":
                r["GW"],

            "Team":
                r["Team"],

            "Opponent":
                r["Opponent"],

            "H/A":
                r["H/A"],

            "API_TAR":
                api_tar,

            "Market_TAR":
                r["Market_TAR"],

            "TAR":
                r["TAR"],

            "API_TDR":
                api_tdr,

            "Market_TDR":
                r["Market_TDR"],

            "TDR":
                r["TDR"],

            "API_TAR_raw":
                safe_round(
                    r["API_TAR_raw"],
                    1
                ),

            "API_TDR_raw":
                safe_round(
                    r["API_TDR_raw"],
                    1
                )
        })

    return tar_tdr_archive


def run_jason_development_94_3B():
    base_url = "https://fantasy.premierleague.com/api/"

    r = requests.get(
        f"{base_url}bootstrap-static/",
        verify=False
    ).json()

    elements = pd.DataFrame(r['elements'])

    # 94.2: Lagets totale GI = sum(goals + assists) fra FPL API.
    elements["Jason_GI"] = (
        pd.to_numeric(elements["goals_scored"], errors="coerce").fillna(0) +
        pd.to_numeric(elements["assists"], errors="coerce").fillna(0)
    )

    team_total_gi_lookup = (
        elements
        .groupby("team")["Jason_GI"]
        .sum()
        .to_dict()
    )

    try:
        next_gw = next(
            event['id']
            for event in r['events']
            if event['is_next']
        )
    except StopIteration:
        next_gw = 1

    gw_cols = [
        f"PS{next_gw}",
        f"PS{next_gw+1}",
        f"PS{next_gw+2}",
        f"PS{next_gw+3}",
        f"PS{next_gw+4}"
    ]

    xp_cols = [
        f"xP{next_gw}",
        f"xP{next_gw+1}",
        f"xP{next_gw+2}",
        f"xP{next_gw+3}",
        f"xP{next_gw+4}"
    ]

    target_gws = list(range(next_gw, next_gw + 5))

    x_min_overrides = read_xmin_overrides_v94(sh)
    market_lookup = read_market_match_lookup_v94()
    xmin_history_archive = read_xmin_history_archive_v94()
    gi_history_archive = read_gi_history_archive_v94()
    xmin_history_archive_rows = []
    gi_history_archive_rows = []
    player_history_lookup = {}
    player_gi_l19_lookup = {}
    team_total_gi_l19_lookup = {}

    teams_det = {
        t['id']: {
            'short': t['short_name'],
            'cs_factor': (
                t['strength_defence_home'] +
                t['strength_defence_away']
            ) / 6000
        }
        for t in r['teams']
    }

    fixtures_r = requests.get(
        f"{base_url}fixtures/",
        verify=False
    ).json()

    teams_lookup = {
        t['id']: t['short_name']
        for t in r['teams']
    }

    update_fdr_overview_worksheet(
        fixtures_r,
        teams_lookup,
        next_gw
    )

    team_strength_lookup = {
        t['id']: {
            "attack_home": t['strength_attack_home'],
            "attack_away": t['strength_attack_away'],
            "defence_home": t['strength_defence_home'],
            "defence_away": t['strength_defence_away']
        }
        for t in r['teams']
    }

    betting_archive = build_betting_data_from_fixtures(
        fixtures_r,
        teams_lookup
    )

    tar_tdr_archive = build_tar_tdr_from_api(
        fixtures_r,
        teams_lookup,
        team_strength_lookup
    )

    # 94.3: Hent spillerhistorikk én gang.
    # Brukes både til xMin, ekte Min 6 og GI Share L19.
    for _, p_pre in elements.iterrows():
        p_pre_id = p_pre['id']
        p_pre_team_id = p_pre['team']

        try:
            summary_pre = requests.get(
                f"{base_url}element-summary/{p_pre_id}/",
                verify=False
            ).json()

            hist_pre = pd.DataFrame(
                summary_pre.get('history', [])
            )
        except:
            hist_pre = pd.DataFrame()

        player_history_lookup[p_pre_id] = hist_pre

        gi_l19 = calculate_player_gi_l19_v94(
            p_pre_id,
            hist_pre,
            next_gw,
            gi_history_archive
        )

        player_gi_l19_lookup[str(p_pre_id)] = gi_l19
        team_total_gi_l19_lookup[p_pre_team_id] = team_total_gi_l19_lookup.get(p_pre_team_id, 0.0) + gi_l19

    # 94.3B: L6 bootstrap. Ved GW1 brukes forrige sesong GW31-36,
    # slik at Form, P6, GI(6), xGI(6), GI/90(6), xGI/90(6) og Heat ikke blir 0.
    l6_prev_rounds, l6_current_rounds, last_6_gws = get_l6_rounds_v94(next_gw)

    main_list = []
    observer_list = []
    season_archive = []
    team_archive = []

    for t in r['teams']:
        team_archive.append({
            'Team': t['name'],
            'Short': t['short_name'],

            'Strength Overall Home': t['strength_overall_home'],
            'Strength Overall Away': t['strength_overall_away'],

            'Strength Attack Home': t['strength_attack_home'],
            'Strength Attack Away': t['strength_attack_away'],

            'Strength Defence Home': t['strength_defence_home'],
            'Strength Defence Away': t['strength_defence_away'],

            'API Attack': round(
                (
                    clean_float(t['strength_attack_home']) +
                    clean_float(t['strength_attack_away'])
                ) / 2,
                1
            ),

            'API Defence': round(
                (
                    clean_float(t['strength_defence_home']) +
                    clean_float(t['strength_defence_away'])
                ) / 2,
                1
            )
        })

    for _, p in elements.iterrows():
        p_id = p['id']
        team_id = p['team']
        pos_id = p['element_type']

        is_injured = p['status'] in ['d', 'i', 's', 'u']

        hist_full = player_history_lookup.get(p_id, pd.DataFrame())

        if hist_full.empty:
            continue

        # Arkiver kamp-for-kamp-minutter så GW25-36 kan brukes etter API reset.
        for _, hr in hist_full.iterrows():
            try:
                xmin_history_archive_rows.append({
                    "Player ID": str(p_id),
                    "Name": f"{p['first_name']} {p['second_name']}",
                    "Club": teams_det[team_id]['short'],
                    "Pos": {
                        1: 'GKP',
                        2: 'DEF',
                        3: 'MID',
                        4: 'FWD'
                    }[pos_id],
                    "Round": int(hr.get("round", 0)),
                    "Minutes": int(hr.get("minutes", 0))
                })

                goals_hr = int(hr.get("goals_scored", 0))
                assists_hr = int(hr.get("assists", 0))

                gi_history_archive_rows.append({
                    "Player ID": str(p_id),
                    "Name": f"{p['first_name']} {p['second_name']}",
                    "Club": teams_det[team_id]['short'],
                    "Pos": {
                        1: 'GKP',
                        2: 'DEF',
                        3: 'MID',
                        4: 'FWD'
                    }[pos_id],
                    "Round": int(hr.get("round", 0)),
                    "Goals": goals_hr,
                    "Assists": assists_hr,
                    "GI": goals_hr + assists_hr
                })
            except:
                pass

        hist_s6 = hist_full[
            hist_full['round'].isin(last_6_gws)
        ]

        m6 = hist_s6['minutes'].sum()

        x6 = round(
            hist_s6['expected_goal_involvements']
            .apply(clean_float)
            .sum(),
            2
        )

        g6 = int(
            hist_s6['goals_scored'].sum() +
            hist_s6['assists'].sum()
        )

        ict6 = round(
            hist_s6['ict_index']
            .apply(clean_float)
            .sum(),
            2
        )

        hist_agg = hist_s6.groupby('round').agg({
            'total_points': 'sum',
            'minutes': 'sum',
            'element': 'count'
        }).to_dict('index')

        f_guide = ""
        min_g = ""
        t_sum = 0

        for gw in last_6_gws:
            if any(
                f['event'] == gw and
                (
                    f['team_h'] == team_id or
                    f['team_a'] == team_id
                )
                for f in fixtures_r
            ):
                d = hist_agg.get(gw)

                if d:
                    t_sum += math.sqrt(
                        max(0, d['total_points'])
                    )

                    num_matches = d.get('element', 1)

                    green_limit = 6 * num_matches
                    yellow_limit = 4 * num_matches

                    f_guide += (
                        "ð¢"
                        if d['total_points'] >= green_limit else
                        "ð¡"
                        if d['total_points'] >= yellow_limit else
                        "ð´"
                        if d['minutes'] > 0 else
                        "âª"
                    )

                    min_g += (
                        "ð¢"
                        if d['minutes'] >= (78 * num_matches) else
                        "ð¡"
                        if d['minutes'] >= (60 * num_matches) else
                        "ð´"
                        if d['minutes'] > 0 else
                        "âª"
                    )
                else:
                    f_guide += "âª"
                    min_g += "âª"
            else:
                f_guide += "X"
                min_g += "X"

        v_form = round((t_sum / 6) * 4, 1)

        team_f_6 = [
            f for f in fixtures_r
            if f['event'] in last_6_gws and
            (
                f['team_h'] == team_id or
                f['team_a'] == team_id
            )
        ]

        min6_real = calculate_min6_real_v94(
            p_id,
            hist_full,
            next_gw,
            xmin_history_archive,
            fixtures_r,
            team_id
        )

        auto_xmin, xmin_l12_minutes, xmin_window = calculate_auto_xmin_l12_v94(
            p_id,
            hist_full,
            next_gw,
            xmin_history_archive
        )

        x_min = x_min_overrides.get(str(p_id), auto_xmin)

        p6_points = int(hist_s6['total_points'].sum())

        ppm_last_6 = (
            round(p6_points / len(team_f_6), 2)
            if len(team_f_6) > 0 else 0.0
        )

        gi_90_6 = (
            round((g6 / m6 * 90), 2)
            if m6 > 0 else 0.0
        )

        xgi_90_6 = (
            round((x6 / m6 * 90), 2)
            if m6 > 0 else 0.0
        )

        season_minutes = (
            float(p['minutes'])
            if float(p['minutes']) > 0 else 1.0
        )

        season_gi = int(
            p['goals_scored'] +
            p['assists']
        )

        season_xgi = round(
            clean_float(
                p['expected_goal_involvements']
            ),
            2
        )

        season_ict = round(
            clean_float(p['ict_index']),
            2
        )

        season_gi_90 = round(
            (season_gi / season_minutes) * 90,
            2
        )

        season_xgi_90 = round(
            (season_xgi / season_minutes) * 90,
            2
        )

        season_ict_90 = round(
            (season_ict / season_minutes) * 90,
            2
        )

        gi_trend = combined_trend_symbol(
            gi_90_6,
            season_gi_90,
            0.15
        )

        xgi_trend = combined_trend_symbol(
            xgi_90_6,
            season_xgi_90,
            0.15
        )

        ict_90_6 = (
            round((ict6 / m6 * 90), 2)
            if m6 > 0 else 0.0
        )

        ict_trend = trend_symbol(
            ict_90_6 - season_ict_90,
            1.5
        )

        heat_score = round(
            (
                (gi_90_6 - season_gi_90) +
                (xgi_90_6 - season_xgi_90)
            ) / 2,
            2
        )

        if x_min < 30:
            heat_symbol = "---"
        else:
            if heat_score >= 0.55:
                heat_symbol = "ð¥ð¥ð¥"
            elif heat_score >= 0.30:
                heat_symbol = "ð¥ð¥"
            elif heat_score >= 0.10:
                heat_symbol = "ð¥"
            elif heat_score > -0.10:
                heat_symbol = "â"
            elif heat_score > -0.30:
                heat_symbol = "ð©"
            else:
                heat_symbol = "ð©ð©"

        xgi_bal = (
            (xgi_90_6 * 0.3) +
            (
                (
                    season_xgi /
                    season_minutes *
                    90
                ) * 0.7
            )
        )

        # Jason 94.0B2 PS-base:
        # PPM er hoveddriver. Form og BxGI/90 er tatt ut av PS.
        # BPS/90 beholdes som liten stabilitetskomponent.
        base_p = (
            float(p['points_per_game']) *
            10.0
        ) + (
            (
                p['bps'] /
                season_minutes *
                90
            ) / 5
        )

        if pos_id == 2:
            base_p += (
                teams_det[team_id]['cs_factor'] *
                22.0
            )

        if (
            (
                p['total_points'] >= 25 or
                next_gw < 20
            ) and
            (m6 >= 90 or next_gw == 1)
        ):
            adj_ps = []
            raw_adj_ps = []

            for gw in target_gws:
                gw_f = [
                    f for f in fixtures_r
                    if f['event'] == gw and
                    (
                        f['team_h'] == team_id or
                        f['team_a'] == team_id
                    )
                ]

                raw_ps_for_gw = sum(
                    base_p *
                    get_fdr_multiplier_v87(
                        float(
                            f['team_h_difficulty']
                            if f['team_h'] == team_id
                            else f['team_a_difficulty']
                        )
                    )
                    for f in gw_f
                )

                raw_adj_ps.append(raw_ps_for_gw)

                adj_ps.append(
                    apply_xmin_to_ps_v94(
                        raw_ps_for_gw,
                        x_min
                    )
                )

            p_dict = {
                'Player ID': p_id,
                'Navn': f"{'â ï¸ ' if is_injured else ''}{p['first_name']} {p['second_name']}",
                'Lag': teams_det[team_id]['short'],
                'Pos': {
                    1: 'GKP',
                    2: 'DEF',
                    3: 'MID',
                    4: 'FWD'
                }[pos_id],
                'Pris': float(p['now_cost']) / 10,
                'Eie %': p['selected_by_percent'],
                'Form': v_form,
                'Siste 6': f_guide,
                'PPK': float(p['points_per_game']),
                'BPS/90': round(
                    (
                        p['bps'] /
                        season_minutes *
                        90
                    ),
                    1
                ),
                'BxGI/90': round(xgi_bal, 2),
                'PS (N5)': round(sum(adj_ps) / 5, 1),
                'PS Tot': round(sum(adj_ps), 1),
                'PS1 Before xMin': round(raw_adj_ps[0], 1) if len(raw_adj_ps) > 0 else 0.0,
                'PS1 After xMin': round(adj_ps[0], 1) if len(adj_ps) > 0 else 0.0,
                'PS(5) Before xMin': round(sum(raw_adj_ps) / 5, 1) if len(raw_adj_ps) > 0 else 0.0,
                'PS(5) After xMin': round(sum(adj_ps) / 5, 1) if len(adj_ps) > 0 else 0.0,
                'PS Tot Before xMin': round(sum(raw_adj_ps), 1),
                'PS Tot After xMin': round(sum(adj_ps), 1),
                'xMin Auto': round(auto_xmin, 1),
                'xMin L12 Min': xmin_l12_minutes,
                'xMin Window': xmin_window,
                'xMin Final': round(x_min, 1),
                'xMin Factor': round(calculate_xmin_factor_v94(x_min), 4)
            }

            for i, name in enumerate(gw_cols):
                p_dict[name] = round(adj_ps[i], 1)

            txp = []

            for i, ps_v in enumerate(adj_ps):
                v = calculate_progressive_xp_v87(
                    ps_v,
                    x_min
                )
                p_dict[xp_cols[i]] = v
                txp.append(v)

            p_dict.update({
                'SUM5': round(sum(txp), 1),
                'P6': p6_points,
                'GI': season_gi,
                'xGI': season_xgi,
                'xGI (S6)': x6,
                'GI (S6)': g6,
                'Pot (6)': round(x6 - g6, 2),
                'GI/90 (6)': gi_90_6,
                'xGI/90 (6)': xgi_90_6,
                'GI/90 (S)': season_gi_90,
                'xGI/90 (S)': season_xgi_90,
                'HeatScore': heat_score,
                'Heat': heat_symbol,
                'Merknad': (
                    p['news']
                    if is_injured else ""
                ),
                'Total min sesong': p['minutes'],
                'Min S/6': min6_real,
                'xMin Auto': round(auto_xmin, 1),
                'xMin L12 Min': xmin_l12_minutes,
                'xMin Window': xmin_window,
                'xMin Final': round(x_min, 1),
                'xMin Factor': round(calculate_xmin_factor_v94(x_min), 4),
                'Min-guide': min_g,
                'Status': "---",
                'GIÎ': gi_trend,
                'xGIÎ': xgi_trend,
                'ICT/90': season_ict_90,
                'ICTÎ': ict_trend
            })

            team_key = normalize_team_key_v94(p_dict["Lag"])
            market_data = market_lookup.get(team_key, {})

            # 94.3: GI Share og Pred GI baseres på rullerende L19.
            gi_l19 = player_gi_l19_lookup.get(str(p_id), 0.0)
            team_total_gi = team_total_gi_l19_lookup.get(team_id, 0.0)

            gi_share = calculate_gi_share_v94(
                gi_l19,
                team_total_gi
            )

            xmin_factor = calculate_xmin_factor_v94(x_min)

            pred_gi = calculate_pred_gi_v94(
                market_data.get("xG", 0),
                gi_l19,
                team_total_gi,
                xmin_factor
            )

            # Baseline for Delta Pred GI:
            # sammenligner L19-Pred GI mot sesongbasert Pred GI.
            team_total_gi_season = team_total_gi_lookup.get(team_id, 0)

            pred_gi_season = calculate_pred_gi_v94(
                market_data.get("xG", 0),
                season_gi,
                team_total_gi_season,
                xmin_factor
            )

            delta_pred_gi = round(pred_gi - pred_gi_season, 2)

            # Legg også market inn i hovedtabellene/posisjonsarkene
            p_dict.update({
                "Team Att": market_data.get("Team Att", 0),
                "Match Att": market_data.get("Match Att", 0),
                "xG": market_data.get("xG", 0),
                "Pred GI": pred_gi,
                "Pred GI Δ": delta_pred_gi,
                "GI Share": gi_share,
                "Team GI": team_total_gi,
                "Team Def": market_data.get("Team Def", 0),
                "Match Def": market_data.get("Match Def", 0),
                "CS%": market_data.get("CS%", 0)
            })

            main_list.append(p_dict)

            observer_list.append({
                "Name": p_dict["Navn"],
                "Club": p_dict["Lag"],
                "Pos": p_dict["Pos"],
                "Price": p_dict["Pris"],
                "xMin": round(x_min, 1),
                "xP": txp[0] if len(txp) > 0 else 0.0,
                "PS(5)": p_dict["PS (N5)"],
                "PS1": round(adj_ps[0], 1) if len(adj_ps) > 0 else 0.0,
                "PS2": round(adj_ps[1], 1) if len(adj_ps) > 1 else 0.0,
                "PS3": round(adj_ps[2], 1) if len(adj_ps) > 2 else 0.0,
                "PS4": round(adj_ps[3], 1) if len(adj_ps) > 3 else 0.0,
                "PS5": round(adj_ps[4], 1) if len(adj_ps) > 4 else 0.0,

                # Market-blokk 94.0F
                "Team Att": market_data.get("Team Att", 0),
                "Match Att": market_data.get("Match Att", 0),
                "xG": market_data.get("xG", 0),
                "Pred GI": pred_gi,
                "Pred GI Δ": delta_pred_gi,
                "GI Share": gi_share,
                "Team GI": team_total_gi,
                "Team Def": market_data.get("Team Def", 0),
                "Match Def": market_data.get("Match Def", 0),
                "CS%": market_data.get("CS%", 0),

                "Heat": heat_symbol,
                "GIÎ": gi_trend,
                "xGIÎ": xgi_trend,
                "BPS/90": p_dict["BPS/90"],
                "ICT/90": season_ict_90,
                "ICTÎ": ict_trend,
                "Form": v_form,
                "PPM LAST 6": ppm_last_6,
                "P6": p6_points,
                "PPM": float(p['points_per_game']),
                "BxGI/90": round(xgi_bal, 2),
                "Mins guide": min_g,
                "Min 6": min6_real,
                "GI": season_gi,
                "xGI": season_xgi,
                "GI (6)": g6,
                "xGI (6)": x6,
                "Pot": round(x6 - g6, 2),
                "GI/90 (6)": gi_90_6,
                "xGI/90 (6)": xgi_90_6,
                "GI/90 (S)": season_gi_90,
                "xGI/90 (S)": season_xgi_90
            })

        season_archive.append({
            'Player ID': p_id,
            'Navn': f"{p['first_name']} {p['second_name']}",
            'Lag': teams_det[team_id]['short'],
            'Pos': {
                1: 'GKP',
                2: 'DEF',
                3: 'MID',
                4: 'FWD'
            }[pos_id],
            'Pris': float(p['now_cost']) / 10,
            'Minutes': p['minutes'],
            'Points': p['total_points'],
            'PPG': p['points_per_game'],
            'GI': season_gi,
            'xGI': season_xgi,
            'GI/90': season_gi_90,
            'xGI/90': season_xgi_90,
            'ICT/90': season_ict_90,
            'BPS': p['bps'],
            'Selected %': p['selected_by_percent']
        })

    df_f = pd.DataFrame(main_list)

    if df_f.empty:
        print("Ingen spillere funnet.")
        return

    order = [
        'Player ID',
        'Navn',
        'Lag',
        'Pos',
        'Pris',
        'Eie %',
        'Form',
        'Siste 6',
        'PPK',
        'BPS/90',
        'BxGI/90',
        'PS (N5)',
        'PS Tot',
        'PS1 Before xMin',
        'PS1 After xMin',
        'PS(5) Before xMin',
        'PS(5) After xMin',
        'PS Tot Before xMin',
        'PS Tot After xMin',
        'xMin Auto',
        'xMin L12 Min',
        'xMin Window',
        'xMin Final',
        'xMin Factor',
        'Team Att',
        'Match Att',
        'xG',
        'Pred GI',
        'Pred GI Δ',
        'GI Share',
        'Team GI',
        'Team Def',
        'Match Def',
        'CS%'
    ] + gw_cols + xp_cols + [
        'SUM5',
        'P6',
        'GI',
        'xGI',
        'xGI (S6)',
        'GI (S6)',
        'Pot (6)',
        'GI/90 (6)',
        'xGI/90 (6)',
        'GI/90 (S)',
        'xGI/90 (S)',
        'HeatScore',
        'Heat',
        'Merknad',
        'Total min sesong',
        'Min S/6',
        'xMin Auto',
        'xMin L12 Min',
        'xMin Window',
        'xMin Final',
        'xMin Factor',
        'Min-guide',
        'Status',
        'GIÎ',
        'xGIÎ',
        'ICT/90',
        'ICTÎ'
    ]

    update_xmin_history_archive_worksheet(xmin_history_archive_rows)
    update_gi_history_archive_worksheet(gi_history_archive_rows)

    update_xmin_override_worksheet(df_f)

    update_worksheet(
        "ICT",
        df_f.sort_values(
            'PS Tot',
            ascending=False
        )[order]
    )

    for s, pos in {
        "Keeper": "GKP",
        "Forsvar": "DEF",
        "Midtbane": "MID",
        "Angrep": "FWD"
    }.items():
        update_worksheet(
            s,
            df_f[df_f['Pos'] == pos]
            .sort_values(
                'PS Tot',
                ascending=False
            )[order]
        )

    update_worksheet(
        "xP",
        df_f.sort_values(
            'SUM5',
            ascending=False
        )[order]
    )

    observer_order = [
        "Name",
        "Club",
        "Pos",
        "Price",

        # Jason-kjerne
        "xMin",
        "xP",
        "PS(5)",
        "PS1",
        "PS2",
        "PS3",
        "PS4",
        "PS5",
        "PPM",
        "Mins guide",
        "Min 6",

        # Market-blokk
        "Team Att",
        "Match Att",
        "xG",
        "Pred GI",
        "Pred GI Δ",
        "GI Share",
        "Team GI",
        "Team Def",
        "Match Def",
        "CS%",

        # Forklaring / temperatur
        "Heat",
        "GIÎ",
        "xGIÎ",
        "BPS/90",
        "ICT/90",
        "ICTÎ",

        # StÃ¸ttemetrikker
        "Form",
        "PPM LAST 6",
        "P6",
        "BxGI/90",

        # RÃ¥data / nerdedata
        "GI",
        "xGI",
        "GI (6)",
        "xGI (6)",
        "Pot",
        "GI/90 (6)",
        "xGI/90 (6)",
        "GI/90 (S)",
        "xGI/90 (S)"
    ]

    df_observer = pd.DataFrame(observer_list)

    update_observer_worksheet(
        df_observer[observer_order]
    )

    update_worksheet(
        "Season_Data",
        pd.DataFrame(season_archive)
    )

    update_worksheet(
        "Team_Data",
        pd.DataFrame(team_archive)
    )

    betting_columns = [
        "GW",
        "Date",
        "HomeTeam",
        "AwayTeam",
        "HomeOdds",
        "DrawOdds",
        "AwayOdds",
        "Source"
    ]

    update_worksheet(
        "Betting_Data",
        pd.DataFrame(
            betting_archive,
            columns=betting_columns
        )
    )

    tar_tdr_columns = [
        "GW",
        "Team",
        "Opponent",
        "H/A",
        "API_TAR",
        "Market_TAR",
        "TAR",
        "API_TDR",
        "Market_TDR",
        "TDR",
        "API_TAR_raw",
        "API_TDR_raw"
    ]

    update_worksheet(
        "TAR_TDR",
        pd.DataFrame(
            tar_tdr_archive,
            columns=tar_tdr_columns
        )
    )

    print(
        "Jason Development 94.3B ferdig "
        "med FDR_Overview, ekte Min 6, L6-bootstrap, L19 GI Share, Pred GI Δ og BPS/90 i TO."
    )


run_jason_development_94_3B()

Market-kobling: 20 lag lest fra Market_Match_Strength_v2_1.
DEBUG MARKET ARS: {'Team Att': 95.0, 'Match Att': 85.0, 'xG': 2.62, 'Team Def': 95.0, 'Match Def': 85.0, 'CS%': 55.1, 'Opp xG': 0.6}
DEBUG MARKET MCI: {'Team Att': 85.05, 'Match Att': 72.56, 'xG': 2.39, 'Team Def': 43.36, 'Match Def': 11.26, 'CS%': 32.5, 'Opp xG': 1.12}
DEBUG MARKET LIV: {'Team Att': 61.37, 'Match Att': 40.38, 'xG': 1.82, 'Team Def': 24.78, 'Match Def': 0.0, 'CS%': 19.2, 'Opp xG': 1.65}
Oppdaterte FDR_Overview med GW1-5.
Oppdaterer xMin_History_Archive med 29747 history-rader...
Oppdaterer GI_History_Archive med 29747 history-rader...
Oppdaterer xMin_Override...
Oppdaterer ICT...
Oppdaterer Keeper...
Oppdaterer Forsvar...
Oppdaterer Midtbane...
Oppdaterer Angrep...
Oppdaterer xP...
Oppdaterer The Observer...


/tmp/ipykernel_1920/1766781992.py:1169: DeprecationWarning: The order of arguments in worksheet.update() has changed. Please pass values first and range_name secondor used named arguments (range_name=, values=)
  ws.update(


Oppdaterer Season_Data...
Oppdaterer Team_Data...
Oppdaterer Betting_Data...
Oppdaterer TAR_TDR...
Jason Development 94.3B ferdig med FDR_Overview, ekte Min 6, L6-bootstrap, L19 GI Share, Pred GI Δ og BPS/90 i TO.
